In [ ]:
import openpyxl
from datetime import datetime, date

DATA_DIR = '/workspace/data/'
wb = openpyxl.load_workbook(DATA_DIR + 'MO14-Purple-City.xlsx', data_only=True)
ws = wb['Data']

# Teams (rows 12-35, col B=team)
teams = []
for r in range(12, 36):
    t = ws.cell(r, 2).value
    if t:
        teams.append(t)

# Squad size (row 29, col D)
squad_size = int(ws.cell(29, 4).value)

# Cost matrix: triangular, rows 13-24 in col C6, headers in row 12 C7-C18
cities_header = [ws.cell(12, c).value for c in range(7, 19)]  # 12 cities

# Row cities (C6, rows 13-24)
row_cities = []
for r in range(13, 25):
    row_cities.append(ws.cell(r, 6).value)

# Build symmetric cost dict from triangular matrix
cost = {}
for idx in range(len(row_cities)):
    rc = row_cities[idx]
    if idx == 0:  # First row (Blue City) has no values
        continue
    for j in range(idx):
        v = ws.cell(14 + idx - 1, 7 + j).value  # row 14+idx-1, col 7+j
        cc = cities_header[j]
        if v is not None:
            cost[(rc, cc)] = v
            cost[(cc, rc)] = v

# Same city = 0
for c in set(x for pair in cost for x in pair):
    cost[(c, c)] = 0

# Games (rows 39-105)
games = []
for r in range(39, 106):
    ta, tb = ws.cell(r, 2).value, ws.cell(r, 3).value
    dt, loc = ws.cell(r, 4).value, ws.cell(r, 5).value
    if ta and tb:
        games.append({'a': ta, 'b': tb, 'date': dt, 'loc': loc})

wb.close()
print(f"Teams: {len(teams)}, Games: {len(games)}, Squad: {squad_size}, Cost entries: {len(cost)}")

In [ ]:
def build_itinerary(team, home, game_list):
    tg = sorted([g for g in game_list if g['a'] == team or g['b'] == team], key=lambda g: g['date'])
    return [home] + [g['loc'] for g in tg] + [home]

def team_cost(team, home, game_list):
    locs = build_itinerary(team, home, game_list)
    return sum(cost.get((locs[i], locs[i+1]), 0) * squad_size
               for i in range(len(locs)-1) if locs[i] != locs[i+1])

def count_flights(team, home, game_list):
    locs = build_itinerary(team, home, game_list)
    return sum(1 for i in range(len(locs)-1) if locs[i] != locs[i+1])

home = 'Purple City'

# Q1: Elephants flights
q1_val = count_flights('Elephants', home, games)
q1 = {4:'A', 5:'B', 6:'C', 7:'D'}[q1_val]

# Q2: Frogs cost
q2_val = team_cost('Frogs', home, games)
q2 = min({'A':15246,'B':16548,'C':24612,'D':25430}, key=lambda k: abs({'A':15246,'B':16548,'C':24612,'D':25430}[k] - q2_val))

# Q3: Total cost
total_all = sum(team_cost(t, home, games) for t in teams)
q3 = min({'A':587223,'B':632499,'C':664713,'D':666414}, key=lambda k: abs({'A':587223,'B':632499,'C':664713,'D':666414}[k] - total_all))

# Q4: Flights TO Lime City
total_lime = 0
for t in teams:
    locs = build_itinerary(t, home, games)
    for i in range(len(locs)-1):
        if locs[i+1] == 'Lime City' and locs[i] != 'Lime City':
            total_lime += cost.get((locs[i], 'Lime City'), 0) * squad_size
q4 = min({'A':46242,'B':47985,'C':48930,'D':49010}, key=lambda k: abs({'A':46242,'B':47985,'C':48930,'D':49010}[k] - total_lime))

# Q5: Flights FROM Blue City
total_blue = 0
for t in teams:
    locs = build_itinerary(t, home, games)
    for i in range(len(locs)-1):
        if locs[i] == 'Blue City' and locs[i+1] != 'Blue City':
            total_blue += cost.get(('Blue City', locs[i+1]), 0) * squad_size
q5 = min({'A':59871,'B':60690,'C':61803,'D':62160}, key=lambda k: abs({'A':59871,'B':60690,'C':61803,'D':62160}[k] - total_blue))

print(f"Q1={q1} ({q1_val}), Q2={q2} ({q2_val}), Q3={q3} ({total_all})")
print(f"Q4={q4} ({total_lime}), Q5={q5} ({total_blue})")

In [ ]:
# Q6: Flights to games May 11-19
total_may = 0
for t in teams:
    tg = sorted([g for g in games if g['a'] == t or g['b'] == t], key=lambda g: g['date'])
    prev = home
    for g in tg:
        gd = g['date'].date() if isinstance(g['date'], datetime) else g['date']
        loc = g['loc']
        if date(2015, 5, 11) <= gd <= date(2015, 5, 19) and prev != loc:
            total_may += cost.get((prev, loc), 0) * squad_size
        prev = loc
q6 = min({'A':126630,'B':127890,'C':128940,'D':129045}, key=lambda k: abs({'A':126630,'B':127890,'C':128940,'D':129045}[k] - total_may))

# Q7: Start/end at Green City
total_green = sum(team_cost(t, 'Green City', games) for t in teams)
q7 = min({'A':626829,'B':663978,'C':675843,'D':716410}, key=lambda k: abs({'A':626829,'B':663978,'C':675843,'D':716410}[k] - total_green))

# Q8: Move Blue City games to Orange City
mod_games = []
for g in games:
    ng = dict(g)
    if ng['loc'] == 'Blue City':
        ng['loc'] = 'Orange City'
    mod_games.append(ng)
total_mod = sum(team_cost(t, home, mod_games) for t in teams)
impact = total_mod - total_all
q8 = min({'A':7917,'B':-9723,'C':-21462,'D':-32256}, key=lambda k: abs({'A':7917,'B':-9723,'C':-21462,'D':-32256}[k] - impact))

print(f"Q6={q6} ({total_may}), Q7={q7} ({total_green}), Q8={q8} ({impact})")

answers = {
    "question1": q1, "question2": q2, "question3": q3, "question4": q4,
    "question5": q5, "question6": q6, "question7": q7, "question8": q8
}
print(f"\nanswers = {answers}")